In [ ]:
from sys import version
import tensorflow
import pandas as pd
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
from keras.optimizers import SGD
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from keras.callbacks import EarlyStopping
from os import makedirs
makedirs("final_nn", exist_ok=True)

print(version)

In [ ]:
# Load and prepare training and xval data
TRAINING_FILE, XVAL_FILE, TEST_FILE = "../combined_data/15min/train.tsv.gz", "../combined_data/15min/xval.tsv.gz", "../combined_data/15min/test.tsv.gz"
train, xval, test = pd.read_csv(TRAINING_FILE, sep='\t'), pd.read_csv(XVAL_FILE, sep='\t'), pd.read_csv(TEST_FILE, sep='\t')
# Remove RapidRide routes.
train = train.loc[train['is_rapid'] == 0.,]
train.to_csv('../combined_data/15min/train_no_rr.tsv.gz', sep='\t', index=False)
xval = xval.loc[xval['is_rapid'] == 0.,]
xval.to_csv('../combined_data/15min/xval_no_rr.tsv.gz', sep='\t', index=False)
test = test.loc[test['is_rapid'] == 0.,]
test.to_csv('../combined_data/15min/test_no_rr.tsv.gz', sep='\t', index=False)

train['hour'] = train['trip_start_hr_15'].apply(lambda x: int(x.split("_")[0]))
xval['hour'] = xval['trip_start_hr_15'].apply(lambda x: int(x.split("_")[0]))
test['hour'] = test['trip_start_hr_15'].apply(lambda x: int(x.split("_")[0]))
print(f'Training dimension: {train.shape}')
print(f'Xval dimension: {xval.shape}')
print(f'Test dimension: {test.shape}')
print('\n'.join(train.columns))
#train.head(n=2).T

In [ ]:
NUM_COLS_TO_SCALE = [
    'orca_total', 
    'frac_disabled', 
    'frac_youth', 
    'frac_senior', 
    'frac_li', 
    'frac_uw',
    'hour'
]
NUM_COLS_OTHER = [
    #'hour'
]
X_CAT_COLS = [
    'is_ns', 
    #'is_rapid', 
    'is_weekend', 
    'trip_start_hr_15',
    'rte', 
    'dir', 
    'day_of_week', 
    'region', 
    'start', 
    'end', 
    'summer'
]

one_hot_encoder = OneHotEncoder()
scaler = StandardScaler()

print("Creating X_train")
X_train = np.concatenate((
    scaler.fit_transform(train[NUM_COLS_TO_SCALE]),
    train[NUM_COLS_OTHER],
    one_hot_encoder.fit_transform(train[X_CAT_COLS]).todense()
), axis=1)

print("Creating X_xval")
X_xval = np.concatenate((
    scaler.transform(xval[NUM_COLS_TO_SCALE]),
    xval[NUM_COLS_OTHER],
    one_hot_encoder.transform(xval[X_CAT_COLS]).todense()
), axis=1)

print("Creating X_test")
X_test = np.concatenate((
    scaler.transform(test[NUM_COLS_TO_SCALE]),
    test[NUM_COLS_OTHER],
    one_hot_encoder.transform(test[X_CAT_COLS]).todense()
), axis=1)

y_train = train['ons']
y_xval = xval['ons']
y_test = test['ons']

np.save("final_nn/preprocessed_15m_X_train.npy", X_train)
np.save("final_nn/preprocessed_15m_X_xval.npy", X_xval)
np.save("final_nn/preprocessed_15m_X_test.npy", X_test)
np.save("final_nn/y_15m_train.npy", y_train)
np.save("final_nn/y_15m_xval.npy", y_xval)
np.save("final_nn/y_15m_test.npy", y_test)

In [ ]:
X_train.shape

In [ ]:
one_hot_encoder.categories_

column_labels = list()
for lab in NUM_COLS_TO_SCALE:
    column_labels.append(f'Scaled: {lab}')

for lab in NUM_COLS_OTHER:
    column_labels.append(f'Numeric: {lab}')

for i, cat in enumerate(X_CAT_COLS):
    for cat_val in one_hot_encoder.categories_[i]:
        column_labels.append(f'{cat}: {cat_val}')

assert len(column_labels) == X_train.shape[1], f"Len of column labels {len(column_labels)} matches dimension of training set {X_train.shape}"

import pickle
with open('final_nn/15m_one_hot_encoder.pkl', 'wb') as f:
    pickle.dump(one_hot_encoder, f)

with open('final_nn/15m_standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('final_nn/15m_column_labels.pkl', 'wb') as f:
    pickle.dump(column_labels, f)

print(column_labels)

In [ ]:
# Find the best possible model using the xval data
from skopt import gp_minimize

def model_fit(params):   # size=600, dropout=0.1, lr=0.2, max_iter=25
    size, dropout, lr = params
    max_iter = 25
    print(f"Fitting model with an initial layer of width {size}, dropout={dropout} and lr={lr}")
    model = Sequential()
    model.add(Dense(size, activation="sigmoid", input_dim=X_train.shape[1]))
    model.add(Dropout(dropout))
    model.add(Dense(1, activation='linear'))
    sgd = SGD(lr=lr, decay=1e-6, momentum=0.9, nesterov=True)
    model.compile(loss='mean_absolute_error', optimizer=sgd)
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, restore_best_weights=True)
    history = model.fit(X_train, y_train, validation_data=(X_xval, y_xval), epochs=max_iter, batch_size=256, callbacks=[es])
    return model.evaluate(X_xval, y_xval)

best = gp_minimize(model_fit, [
        (200, 1200),  # size
        (0.1, 0.5),   # dropout
        (0.1, 1)      # lr
    ],
    n_calls=25,
    n_random_starts=3
)

In [ ]:
model = Sequential()
model.add(Dense(1020, activation="sigmoid", input_dim=X_train.shape[1]))
model.add(Dropout(0.23544620354920223))
model.add(Dense(1, activation='linear'))
sgd = SGD(lr=0.16287174712352556, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='mean_absolute_error', optimizer=sgd)
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_xval, y_xval), epochs=25, batch_size=256, callbacks=[es])

In [ ]:
from keras.models import model_from_json

with open("final_nn/model_15min.json", "r") as json_file:
    model = model_from_json(json_file.read())
sgd = SGD(lr=0.16287174712352556, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='mean_absolute_error', optimizer=sgd)
model.load_weights("final_nn/model_15min_weights_train.h5")
print(f'Test perf: {model.evaluate(X_test, y_test, batch_size=128)}')

In [ ]:
model_json = model.to_json()
with open("final_nn/model_15min.json", "w") as json_file:
    json_file.write(model_json)

model.save_weights("final_nn/model_15min_weights_train.h5")

In [ ]:
# # Create test set predictions
# from keras.models import model_from_json

# with open("final_nn/model_15min.json", "r") as json_file:
#     model = model_from_json(json_file.read())

# model.load_weights("final_nn/model_15min_weights_train.h5")

with open("../predictions/final_nn_15min_test.txt", 'wt') as f:
    for val in model.predict(X_test).squeeze():
        f.write(f'{val:.15f}\n')

with open("../predictions/final_nn_15min_xval.txt", 'wt') as f:
    for val in model.predict(X_xval).squeeze():
        f.write(f'{val:.15f}\n')

In [ ]:
# Fit another model on 30m data.
# Load and prepare training and xval data
TRAINING_FILE, XVAL_FILE, TEST_FILE = "../combined_data/30min/train.tsv.gz", "../combined_data/30min/xval.tsv.gz", "../combined_data/30min/test.tsv.gz"
train, xval, test = pd.read_csv(TRAINING_FILE, sep='\t'), pd.read_csv(XVAL_FILE, sep='\t'), pd.read_csv(TEST_FILE, sep='\t')
# Remove RapidRide routes.
train = train.loc[train['is_rapid'] == 0.,]
train.to_csv('../combined_data/30min/train_no_rr.tsv.gz', sep='\t', index=False)
xval = xval.loc[xval['is_rapid'] == 0.,]
xval.to_csv('../combined_data/30min/xval_no_rr.tsv.gz', sep='\t', index=False)
test = test.loc[test['is_rapid'] == 0.,]
test.to_csv('../combined_data/30min/test_no_rr.tsv.gz', sep='\t', index=False)

train['hour'] = train['trip_start_hr_30'].apply(lambda x: int(x.split("_")[0]))
xval['hour'] = xval['trip_start_hr_30'].apply(lambda x: int(x.split("_")[0]))
test['hour'] = test['trip_start_hr_30'].apply(lambda x: int(x.split("_")[0]))

X_NUM_COLS = [
    'orca_total', 
    'frac_disabled', 
    'frac_youth', 
    'frac_senior', 
    'frac_li', 
    'frac_uw',
    'hour'
]
X_CAT_COLS = [
    'is_ns', 
    'is_rapid', 
    'is_weekend', 
    'trip_start_hr_30',
    'rte', 
    'dir', 
    'day_of_week', 
    'region', 
    'start', 
    'end', 
    'summer'
]

one_hot_encoder = OneHotEncoder()
scaler = StandardScaler()

print("Creating X_train")
X_train = np.concatenate((
    scaler.fit_transform(train[X_NUM_COLS]),
    one_hot_encoder.fit_transform(train[X_CAT_COLS]).todense()
), axis=1)

print("Creating X_xval")
X_xval = np.concatenate((
    scaler.transform(xval[X_NUM_COLS]),
    one_hot_encoder.transform(xval[X_CAT_COLS]).todense()
), axis=1)

print("Creating X_test")
X_test = np.concatenate((
    scaler.transform(test[X_NUM_COLS]),
    one_hot_encoder.transform(test[X_CAT_COLS]).todense()
), axis=1)

y_train = train['ons']
y_xval = xval['ons']
y_test = test['ons']

np.save("final_nn/preprocessed_30m_X_train.npy", X_train)
np.save("final_nn/preprocessed_30m_X_xval.npy", X_xval)
np.save("final_nn/preprocessed_30m_X_test.npy", X_test)
np.save("final_nn/y_30m_train.npy", y_train)
np.save("final_nn/y_30m_xval.npy", y_xval)
np.save("final_nn/y_30m_test.npy", y_test)

In [ ]:
column_labels = list()
for lab in X_NUM_COLS:
    column_labels.append(f'num: {lab}')

for i, cat in enumerate(X_CAT_COLS):
    for cat_val in one_hot_encoder.categories_[i]:
        column_labels.append(f'{cat}: {cat_val}')

assert len(column_labels) == X_train.shape[1], f"Len of column labels {len(column_labels)} matches dimension of training set {X_train.shape}"

import pickle
with open('final_nn/30m_one_hot_encoder.pkl', 'wb') as f:
    pickle.dump(one_hot_encoder, f)

with open('final_nn/30m_standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('final_nn/30m_column_labels.pkl', 'wb') as f:
    pickle.dump(column_labels, f)


In [ ]:
# Find the best possible model using the xval data
from skopt import gp_minimize

def model_fit(params):
    size, dropout, lr = params
    max_iter = 25
    print(f"Fitting model with an initial layer of width {size}, dropout={dropout} and lr={lr}")
    model = Sequential()
    model.add(Dense(size, activation="sigmoid", input_dim=X_train.shape[1]))
    model.add(Dropout(dropout))
    model.add(Dense(1, activation='linear'))
    sgd = SGD(lr=lr, decay=1e-6, momentum=0.9, nesterov=True)
    model.compile(loss='mean_absolute_error', optimizer=sgd)
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, restore_best_weights=True)
    history = model.fit(X_train, y_train, validation_data=(X_xval, y_xval), epochs=max_iter, batch_size=256, callbacks=[es])
    return model.evaluate(X_xval, y_xval)

best = gp_minimize(model_fit, [
        (800, 2000),  # size
        (0.1, 0.5),   # dropout
        (0.1, 1)      # lr
    ],
    n_calls=25,
    n_random_starts=5
)

In [ ]:
# After running the above step on cloud compute, the best model I found was this:
# 2000, dropout=0.1 and lr=0.1
model = Sequential()
model.add(Dense(2000, activation="sigmoid", input_dim=X_train.shape[1]))
model.add(Dropout(0.1))
model.add(Dense(1, activation='linear'))
sgd = SGD(lr=0.1, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='mean_absolute_error', optimizer=sgd)
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, restore_best_weights=True)
model.fit(X_train, y_train, validation_data=(X_xval, y_xval), epochs=25, batch_size=256, callbacks=[es])
model.evaluate(X_test, y_test)

In [ ]:
model_json = model.to_json()
with open("final_nn/model_30min.json", "w") as json_file:
    json_file.write(model_json)

model.save_weights("final_nn/model_30min_weights_train.h5")

In [ ]:
# Create test set predictions out of the better performing model
from keras.models import model_from_json

#with open("final_nn/model_30min.json", "r") as json_file:
#    model = model_from_json(json_file.read())

#model.load_weights("final_nn/model_30min_weights_train.h5")

with open("../predictions/final_nn_30min_test.txt", 'wt') as f:
    for val in model.predict(X_test).squeeze():
        f.write(f'{val:.15f}\n')

with open("../predictions/final_nn_30min_xval.txt", 'wt') as f:
    for val in model.predict(X_xval).squeeze():
        f.write(f'{val:.15f}\n')

In [ ]:
# And for 1hr

TRAINING_FILE, XVAL_FILE, TEST_FILE = "../combined_data/hr/train.tsv.gz", "../combined_data/hr/xval.tsv.gz", "../combined_data/hr/test.tsv.gz"
train, xval, test = pd.read_csv(TRAINING_FILE, sep='\t'), pd.read_csv(XVAL_FILE, sep='\t'), pd.read_csv(TEST_FILE, sep='\t')
# Remove RapidRide routes.
train = train.loc[train['is_rapid'] == 0.,]
train.to_csv('../combined_data/hr/train_no_rr.tsv.gz', sep='\t', index=False)
xval = xval.loc[xval['is_rapid'] == 0.,]
xval.to_csv('../combined_data/hr/xval_no_rr.tsv.gz', sep='\t', index=False)
test = test.loc[test['is_rapid'] == 0.,]
test.to_csv('../combined_data/hr/test_no_rr.tsv.gz', sep='\t', index=False)

train['hour'] = train['trip_start_hr'].apply(int)
xval['hour'] = xval['trip_start_hr'].apply(int)
test['hour'] = test['trip_start_hr'].apply(int)
print(f'Training dimension: {train.shape}')
print(f'Xval dimension: {xval.shape}')
print(f'Test dimension: {test.shape}')

X_NUM_COLS = [
    'orca_total', 
    'frac_disabled', 
    'frac_youth', 
    'frac_senior', 
    'frac_li', 
    'frac_uw',
    'hour'
]
X_CAT_COLS = [
    'is_ns', 
    'is_rapid', 
    'is_weekend', 
    'trip_start_hr',
    'rte', 
    'dir', 
    'day_of_week', 
    'region', 
    'start', 
    'end', 
    'summer'
]

one_hot_encoder = OneHotEncoder()
scaler = StandardScaler()

print("Creating X_train")
X_train = np.concatenate((
    scaler.fit_transform(train[X_NUM_COLS]),
    one_hot_encoder.fit_transform(train[X_CAT_COLS]).todense()
), axis=1)

print("Creating X_xval")
X_xval = np.concatenate((
    scaler.transform(xval[X_NUM_COLS]),
    one_hot_encoder.transform(xval[X_CAT_COLS]).todense()
), axis=1)

print("Creating X_test")
X_test = np.concatenate((
    scaler.transform(test[X_NUM_COLS]),
    one_hot_encoder.transform(test[X_CAT_COLS]).todense()
), axis=1)

y_train = train['ons']
y_xval = xval['ons']
y_test = test['ons']

np.save("final_nn/preprocessed_hr_X_train.npy", X_train)
np.save("final_nn/preprocessed_hr_X_xval.npy", X_xval)
np.save("final_nn/preprocessed_hr_X_test.npy", X_test)
np.save("final_nn/y_hr_train.npy", y_train)
np.save("final_nn/y_hr_xval.npy", y_xval)
np.save("final_nn/y_hr_test.npy", y_test)

In [ ]:
column_labels = list()
for lab in X_NUM_COLS:
    column_labels.append(f'num: {lab}')

for i, cat in enumerate(X_CAT_COLS):
    for cat_val in one_hot_encoder.categories_[i]:
        column_labels.append(f'{cat}: {cat_val}')

assert len(column_labels) == X_train.shape[1], f"Len of column labels {len(column_labels)} matches dimension of training set {X_train.shape}"

import pickle
with open('final_nn/hr_one_hot_encoder.pkl', 'wb') as f:
    pickle.dump(one_hot_encoder, f)

with open('final_nn/hr_standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('final_nn/hr_column_labels.pkl', 'wb') as f:
    pickle.dump(column_labels, f)


In [ ]:
from skopt import gp_minimize

def model_fit(params):
    size, dropout, lr = params
    max_iter = 25
    print(f"Fitting model with an initial layer of width {size}, dropout={dropout} and lr={lr}")
    model = Sequential()
    model.add(Dense(size, activation="sigmoid", input_dim=X_train.shape[1]))
    model.add(Dropout(dropout))
    model.add(Dense(1, activation='linear'))
    sgd = SGD(lr=lr, decay=1e-6, momentum=0.9, nesterov=True)
    model.compile(loss='mean_absolute_error', optimizer=sgd)
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, restore_best_weights=True)
    history = model.fit(X_train, y_train, validation_data=(X_xval, y_xval), epochs=max_iter, batch_size=256, callbacks=[es])
    return model.evaluate(X_xval, y_xval)

best = gp_minimize(model_fit, [
        (800, 2000),  # size
        (0.1, 0.5),   # dropout
        (0.1, 1)      # lr
    ],
    n_calls=25,
    n_random_starts=5
)

In [ ]:
model = Sequential()
model.add(Dense(2000, activation="sigmoid", input_dim=X_train.shape[1]))
model.add(Dropout(0.1))
model.add(Dense(1, activation='linear'))
sgd = SGD(lr=0.22902961281, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='mean_absolute_error', optimizer=sgd)
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, restore_best_weights=True)
model.fit(X_train, y_train, validation_data=(X_xval, y_xval), epochs=25, batch_size=256, callbacks=[es])

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
model_json = model.to_json()
with open("final_nn/model_hr.json", "w") as json_file:
    json_file.write(model_json)

model.save_weights("final_nn/model_hr_weights_train.h5")

In [ ]:
# Create test set predictions out of the better performing model
from keras.models import model_from_json

# with open("final_nn/model_hr.json", "r") as json_file:
#     model = model_from_json(json_file.read())

# model.load_weights("final_nn/model_hr_weights_train.h5")

with open("../predictions/final_nn_hr_test.txt", 'wt') as f:
    for val in model.predict(X_test).squeeze():
        f.write(f'{val:.15f}\n')

with open("../predictions/final_nn_hr_xval.txt", 'wt') as f:
    for val in model.predict(X_xval).squeeze():
        f.write(f'{val:.15f}\n')

In [ ]:
from matplotlib import pyplot as plt
plt.style.use('seaborn')
d = pd.read_csv("../evaluation/15m_test_group_perf.csv")
d['lab'] = d['group'].apply(str) + '\nn = ' + d['n'].apply(str)
d = d.set_index(['lab'])
plt.figure(figsize=(5,15))
d['mae'].plot.barh()
plt.title("Test set performance by route type")
plt.xlabel("Mean absolute error")
plt.ylabel("")
plt.show()

In [ ]:
d